# Microsoft Agent Framework — Analyst → Strategist workflow

Quick correction to tonight's slide: AutoGen's actual replacement isn't Semantic
Kernel — Microsoft merged **both** AutoGen and Semantic Kernel into a single SDK,
**Microsoft Agent Framework (MAF)**, which reached GA in April 2026. It keeps AutoGen's
multi-agent conversation patterns and Semantic Kernel's enterprise plumbing (plugins,
middleware, observability), with native support for Anthropic, OpenAI, Azure, Bedrock,
and others.

| Framework | Core mental model | How multi-step flow is defined | Orchestration style |
|---|---|---|---|
| LangGraph | Explicit state graph | You wire nodes (functions/LLM calls) and edges yourself | Manual, graph-based — full control, more code |
| CrewAI | Role-based crew | You define `Agent`s (role/goal) and `Task`s; a `Process` (sequential/hierarchical) runs them | Opinionated, role-based — less code, less control |
| **Microsoft Agent Framework** | Unified successor to AutoGen + Semantic Kernel | Agents come from a chat client; you wire them with a Builder (Sequential, Concurrent, Handoff, GroupChat, Magentic) | Both — low-level like LangGraph, or pre-built patterns like CrewAI |



In [ ]:
%pip install -q agent-framework-core==1.18.0 agent-framework-anthropic==1.0.0b260910 agent-framework-orchestrations==1.1.1

> **Gotcha:** After the install cell finishes for the first time, use
> **Runtime ▸ Restart session** before running the rest of the notebook. Colab can keep
> an old, incompatible copy of a dependency loaded in memory even after a fresh install
> replaces it on disk.

> MAF's provider packages are still on beta version numbers and move fast — the pins above are the versions this notebook was tested against, request *and* response paths. An earlier draft of this notebook pinned an older `agent-framework-anthropic` (from the `agent-framework` metapackage's own default) that constructs its response object with a keyword argument current `agent-framework-core` no longer accepts — it looks fine until a real API call succeeds, then fails on `ChatResponse.__init__() got an unexpected keyword argument 'model_id'`. If you see that error, it means your installed `agent-framework-anthropic` predates this fix — rerun the install cell to get the pinned version above. An unpinned `pip install agent-framework` also works but pulls in ~20 extra provider packages (Azure, Bedrock, Gemini, etc.) you don't need for this demo.


In [ ]:
import os

def _load(key, default=None):
    """Colab Secrets first, then environment variable, then interactive prompt."""
    try:
        from google.colab import userdata
        val = userdata.get(key)
        if val:
            return val
    except Exception:
        pass
    val = os.environ.get(key, default)
    if not val:
        from getpass import getpass
        val = getpass(f"{key}: ")
    return val

ANTHROPIC_API_KEY = _load("ANTHROPIC_API_KEY")
ANTHROPIC_MODEL = _load("ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929")
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

print(f"Using model: {ANTHROPIC_MODEL}")


## The scenario

An assistant reads user interviews, support tickets, and
competitor notes, and is supposed to turn that into a feature recommendation — without
mismatched analysis or hallucinated suggestions.

We'll build that as a **two-step pipeline**:

1. **Analyst** — reads the raw signals, extracts 2–3 themes, and cites which signal(s)
   support each one (a small guardrail against fabrication).
2. **Strategist** — takes *only* the Analyst's themes and recommends ONE feature with a
   one-line rationale. It's told not to invent anything the Analyst didn't surface.

Same signals, same two-step shape, three different frameworks — that's the point of
building all three tonight.

In MAF terms: two **agents** built from one `AnthropicClient`, wired with a **`SequentialBuilder`** — a pre-built workflow pattern, so there's no manual graph to draw.


In [ ]:
SIGNALS = """
- User interview (Maria, SMB owner): "I got lost during setup and gave up before finishing."
- Support ticket #4821: "Can't find where to invite my team members. Confusing menu."
- Competitor teardown (RivalApp): just shipped a guided setup checklist with a progress bar.
- Support ticket #4903: "Wish there was a quick tour when I first logged in."
""".strip()

print(SIGNALS)


In [ ]:
from agent_framework.anthropic import AnthropicClient

client = AnthropicClient(model=ANTHROPIC_MODEL)

analyst = client.as_agent(
    name="Analyst",
    instructions=(
        "You are a product signal analyst. Extract 2-3 themes from the signals "
        "you're given as bullet points. Cite which signal supports each theme."
    ),
)

strategist = client.as_agent(
    name="Strategist",
    instructions=(
        "You are a product strategist. Using ONLY the themes you're given, "
        "recommend ONE feature to prioritize next sprint with a one-line rationale. "
        "Do not invent new signals."
    ),
)


In [ ]:
from agent_framework.orchestrations import SequentialBuilder

workflow = SequentialBuilder(participants=[analyst, strategist]).build()

result = await workflow.run(f"Signals:\n{SIGNALS}")
for output in result.get_outputs():
    print(getattr(output, "text", output))
    print("---")


## How Microsoft Agent Framework differs from LangGraph and CrewAI

| Framework | Core mental model | How multi-step flow is defined | Orchestration style |
|---|---|---|---|
| LangGraph | Explicit state graph | You wire nodes (functions/LLM calls) and edges yourself | Manual, graph-based — full control, more code |
| CrewAI | Role-based crew | You define `Agent`s (role/goal) and `Task`s; a `Process` (sequential/hierarchical) runs them | Opinionated, role-based — less code, less control |
| **Microsoft Agent Framework** | Unified successor to AutoGen + Semantic Kernel | Agents come from a chat client; you wire them with a Builder (Sequential, Concurrent, Handoff, GroupChat, Magentic) | Both — low-level like LangGraph, or pre-built patterns like CrewAI |


**Networking analogy:** if LangGraph is hand-cabling static routes and CrewAI is a
NOC runbook, MAF is the SD-WAN overlay controller — instead of wiring every link or
writing pure job descriptions, you pick a pre-built topology template
(`SequentialBuilder`, `HandoffBuilder`, `GroupChatBuilder`, `MagenticBuilder`) the way
an SD-WAN controller hands you hub-and-spoke or full-mesh overlays instead of manual
point-to-point tunnels. It also still has the low-level `WorkflowBuilder` underneath, so
you can drop down to LangGraph-style manual wiring when a template doesn't fit.

**Honest caveat:** MAF is the youngest framework in this lesson — GA for about five
months, versus years of production mileage for LangGraph and CrewAI. Treat it as
promising and worth watching, not yet the safe default for a production system the way
the other two are.

**Try it yourself:** swap `SequentialBuilder` for `HandoffBuilder` and give the
Strategist a way to hand the conversation back to the Analyst if it decides it needs
more themes.
